# Módulo 11 · Práctica: entrenar redes profundas

Vas a construir una red de 24 capas, verla fallar y arreglarla con las mismas piezas que usa un LLM: inicialización, conexiones residuales, RMSNorm, recorte de gradiente y el formato numérico adecuado. Todo en CPU con PyTorch y datos sintéticos (no hay descargas).

| Parte | Qué haces |
| --- | --- |
| 0 | Preparación y datos sintéticos |
| 1 | Formatos numéricos: fp32, fp16, bf16, fp8 |
| 2 | Overflow en fp16 frente a bf16 y log-sum-exp estable |
| 3 | Red plana de 24 capas: histogramas y gradientes |
| 4 | RMSNorm desde cero |
| 5 | Red residual pre-norm |
| 6 | Un entrenamiento roto: NaN y recorte de gradiente |
| 7 | Dropout invertido |
| — | Soluciones de referencia |

**Cómo trabajar:** ejecuta en orden. Completa las celdas `# TU CÓDIGO` y ejecuta la `# Comprobación` que las sigue.

## Parte 0 · Preparación

Usamos un solo hilo: en máquinas pequeñas las matrices de 64×64 van más rápido así. Los datos son 512 puntos en 16 dimensiones etiquetados en 4 clases por una red «profesora» aleatoria.

In [ ]:
import math, time
import torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt
torch.set_num_threads(1)
torch.manual_seed(0)

N, D, C = 512, 16, 4          # ejemplos, dimensión de entrada, clases
L, W = 24, 64                 # capas ocultas y anchura
X = torch.randn(N, D)
profesora = nn.Sequential(nn.Linear(D, 32), nn.Tanh(), nn.Linear(32, C))
with torch.no_grad():
    y = profesora(X).argmax(1)
print("Ejemplos por clase:", torch.bincount(y).tolist())
print("Pérdida de adivinar al azar: ln 4 =", round(math.log(4), 3))

## Parte 1 · Formatos numéricos

Un número en coma flotante con $e$ bits de exponente y $m$ de mantisa vale $(-1)^s\,(1{,}f)_2\cdot 2^{E-\text{sesgo}}$, con sesgo $=2^{e-1}-1$. De ahí:

$$\text{máx} = (2-2^{-m})\,2^{\text{sesgo}},\qquad \varepsilon = 2^{-m},\qquad \text{mín. normal} = 2^{1-\text{sesgo}}$$

(El exponente más alto se reserva para inf y NaN.)

**Ejercicio 1.** Escribe `formato(e, m)` que devuelva `(maximo, eps, min_normal)`.

In [ ]:
# TU CÓDIGO
def formato(e, m):
    return None, None, None

In [ ]:
# Comprobación
for nombre, dt, e, m in [("fp32", torch.float32, 8, 23), ("fp16", torch.float16, 5, 10),
                         ("bf16", torch.bfloat16, 8, 7), ("fp8 e5m2", torch.float8_e5m2, 5, 2)]:
    mx, eps, mn = formato(e, m)
    fi = torch.finfo(dt)
    assert math.isclose(mx, fi.max) and eps == fi.eps and mn == fi.smallest_normal, f"Fallo en {nombre}"
    print(f"{nombre:9s} máx={mx:.4g}  eps={eps:.4g}  mín normal={mn:.4g}")
print("Correcto. Ojo: fp8 e4m3fn no sigue la regla (máx 448), porque solo reserva un patrón para NaN:",
      torch.finfo(torch.float8_e4m3fn).max)

## Parte 2 · Overflow en fp16 frente a bf16

fp16 tiene más precisión que bf16 pero muchísimo menos rango. Observa qué pasa con un producto y una exponencial moderados.

In [ ]:
for dt in [torch.float32, torch.float16, torch.bfloat16]:
    a = torch.tensor(300., dtype=dt)
    print(f"{str(dt):15s} 300·300 = {(a*a).item():<10}  e^12 = {torch.exp(torch.tensor(12., dtype=dt)).item():<10}"
          f"  1 + 2^-8 = {(torch.tensor(1., dtype=dt) + 2**-8).item()}")
z = torch.tensor([12., 10., 0.], dtype=torch.float16)
print("Softmax ingenua en fp16:", (torch.exp(z) / torch.exp(z).sum()).tolist())

La softmax ingenua da `nan` porque $e^{12}\approx 162.755 > 65.504$ se convierte en `inf` e $\infty/\infty$ = NaN. La solución es restar el máximo:

$$\log\sum_i e^{z_i} = c + \log\sum_i e^{z_i - c},\qquad c=\max_i z_i$$

**Ejercicio 2.** Implementa `lse(z)` estable para un vector 1D.

In [ ]:
# TU CÓDIGO
def lse(z):
    return None

In [ ]:
# Comprobación
z32 = torch.tensor([1000., 999., 998.])
assert torch.isclose(lse(z32), torch.logsumexp(z32, 0)), "No coincide con torch.logsumexp"
assert torch.isfinite(lse(z.float())) and torch.isfinite(lse(z)), "Debe ser finito incluso en fp16"
print("Correcto. lse([1000, 999, 998]) =", round(lse(z32).item(), 4))
print("Softmax estable en fp16:", torch.exp(z - lse(z)).tolist())

## Parte 3 · Una red plana de 24 capas

Cada capa es `tanh(W h + b)`. La inicialización por defecto de `nn.Linear` es uniforme en ±1/√n, con varianza 1/(3n). Guardamos las activaciones para verlas.

In [ ]:
class RedPlana(nn.Module):
    def __init__(self, std=None):
        super().__init__()
        self.entrada = nn.Linear(D, W)
        self.capas = nn.ModuleList([nn.Linear(W, W) for _ in range(L)])
        self.salida = nn.Linear(W, C)
        if std is not None:                      # inicialización gaussiana a medida
            for c in self.capas:
                nn.init.normal_(c.weight, 0, std); nn.init.zeros_(c.bias)
    def forward(self, x):
        h = self.entrada(x)
        self.acts = []
        for capa in self.capas:
            h = torch.tanh(capa(h))
            self.acts.append(h)
        return self.salida(h)

def histogramas(modelo, titulo, capas=(0, 7, 15, 23)):
    with torch.no_grad():
        modelo(X)
    fig, ax = plt.subplots(1, len(capas), figsize=(12, 2.2), sharey=True)
    for a, k in zip(ax, capas):
        v = modelo.acts[k].flatten()
        a.hist(v.numpy(), bins=50); a.set_title(f"capa {k+1}: std={v.std():.3f}", fontsize=9)
    fig.suptitle(titulo); plt.tight_layout(); plt.show()

def entrenar(modelo, pasos=120, lr=0.1):
    opt = torch.optim.SGD(modelo.parameters(), lr=lr)
    hist = []
    for _ in range(pasos):
        loss = F.cross_entropy(modelo(X), y)
        opt.zero_grad(); loss.backward(); opt.step()
        hist.append(loss.item())
    return hist

**Ejercicio 3.** Escribe `normas_grad(modelo)`: hace una pasada hacia delante y atrás con la entropía cruzada y devuelve la lista de normas `capa.weight.grad.norm()` de las 24 capas ocultas.

In [ ]:
# TU CÓDIGO
def normas_grad(modelo):
    return []

In [ ]:
# Comprobación
torch.manual_seed(1); plana = RedPlana()
g = normas_grad(plana)
assert len(g) == L, "Debe haber una norma por capa oculta"
print(f"capa 1: {g[0]:.2e}   capa 24: {g[-1]:.2e}   cociente: {g[0]/g[-1]:.1e}")
assert g[0] / g[-1] < 1e-3, "En la red plana el gradiente de la primera capa debería ser diminuto"
print("Correcto: el gradiente se desvanece. Estimación del módulo: (1/√3)^24 =", f"{3**-12:.1e}")

In [ ]:
torch.manual_seed(1); grande = RedPlana(std=0.5)
torch.manual_seed(1); xavier = RedPlana(std=1/math.sqrt(W))
histogramas(plana, "Por defecto (var 1/3n): las activaciones se apagan")
histogramas(grande, "std = 0,5: tanh saturada en ±1")
histogramas(xavier, "std = 1/√n: escala estable")
for nombre, m in [("defecto", plana), ("grande", grande), ("xavier", xavier)]:
    gg = normas_grad(m)
    print(f"{nombre:8s} grad capa1={gg[0]:.2e} capa24={gg[-1]:.2e}  saturadas={(m.acts[-1].abs() > 0.97).float().mean():.2f}")

**Diagnóstico.** Histogramas que se estrechan hacia 0 → gradiente que se desvanece. Histogramas con dos picos en ±1 → saturación: la derivada de tanh es casi 0 y el gradiente de las primeras capas se vuelve errático. Escala estable → la red aprende. Entrénalas:

In [ ]:
t0 = time.time()
torch.manual_seed(1); h_plana = entrenar(RedPlana())
torch.manual_seed(1); h_xav = entrenar(RedPlana(std=1/math.sqrt(W)))
print(f"plana: {h_plana[0]:.3f} -> {h_plana[-1]:.3f}   xavier: {h_xav[0]:.3f} -> {h_xav[-1]:.3f}   ({time.time()-t0:.0f} s)")

## Parte 4 · RMSNorm desde cero

$$\operatorname{RMSNorm}(x) = g\odot\frac{x}{\sqrt{\tfrac1d\sum_i x_i^2+\epsilon}}$$

**Ejercicio 4.** Implementa `MiRMSNorm(d, eps=1e-6)` con un parámetro `g` inicializado a unos.

In [ ]:
# TU CÓDIGO
class MiRMSNorm(nn.Module):
    def __init__(self, d, eps=1e-6):
        super().__init__()
    def forward(self, x):
        return x

In [ ]:
# Comprobación
x = torch.tensor([[2., 4., 6.]])
mia, ref = MiRMSNorm(3), nn.RMSNorm(3, eps=1e-6)
assert torch.allclose(mia(x), ref(x), atol=1e-5), "No coincide con nn.RMSNorm"
assert any(p.requires_grad for p in mia.parameters()), "g debe ser un nn.Parameter"
print("Correcto:", mia(x).detach().numpy().round(3), "  RMS de la salida:", round(mia(x).pow(2).mean().sqrt().item(), 3))

## Parte 5 · Red residual pre-norm

El bloque de un LLM moderno es $h \leftarrow h + F(\operatorname{Norm}(h))$: normalizas **dentro** de la rama y la autopista $h$ queda intacta. Aquí $F(u)=\tanh(Wu+b)$.

**Ejercicio 5.** Implementa `RedResidual` con `entrada`, 24 `capas` lineales, 24 `normas` (`MiRMSNorm`), una `norma_final` antes de la `salida`, y guarda cada $h$ en `self.acts`. Usa la inicialización por defecto (la misma que hundió a la red plana).

In [ ]:
# TU CÓDIGO
class RedResidual(nn.Module):
    def __init__(self):
        super().__init__()
    def forward(self, x):
        return None

In [ ]:
# Comprobación
torch.manual_seed(1); res = RedResidual()
g = normas_grad(res)
print(f"residual  grad capa1={g[0]:.2e}  capa24={g[-1]:.2e}  cociente={g[0]/g[-1]:.2f}")
assert 0.1 < g[0] / g[-1] < 10, "Con residuos el gradiente debería llegar a la capa 1 con tamaño parecido"
t0 = time.time(); torch.manual_seed(1); h_res = entrenar(res)
print(f"pérdida: {h_res[0]:.3f} -> {h_res[-1]:.3f}  ({time.time()-t0:.0f} s)")
assert h_res[-1] < 0.3 and h_plana[-1] > 1.2, "La residual debe aprender y la plana no"
plt.figure(figsize=(6, 3)); plt.plot(h_plana, label="plana"); plt.plot(h_xav, label="plana + Xavier")
plt.plot(h_res, label="residual + RMSNorm"); plt.xlabel("paso"); plt.ylabel("pérdida"); plt.legend(); plt.show()
print("Correcto")

## Parte 6 · Un entrenamiento roto

Un compañero sube la tasa de aprendizaje a 10 «para ir más rápido» en una red residual **sin normalización**. Ejecuta y observa:

In [ ]:
class ResidualSinNorma(nn.Module):
    def __init__(self):
        super().__init__()
        self.entrada = nn.Linear(D, W)
        self.capas = nn.ModuleList([nn.Linear(W, W) for _ in range(L)])
        self.salida = nn.Linear(W, C)
    def forward(self, x):
        h = self.entrada(x)
        for capa in self.capas:
            h = h + torch.tanh(capa(h))
        return self.salida(h)

torch.manual_seed(1); roto = ResidualSinNorma()
h_roto = entrenar(roto, pasos=60, lr=10.0)
print("pérdida cada 10 pasos:", [f"{v:.3g}" for v in h_roto[::10]])
primer_nan = next(i for i, v in enumerate(h_roto) if not math.isfinite(v))
print("Primer paso con pérdida no finita:", primer_nan)

La pérdida crece varios órdenes de magnitud y acaba en `nan`: un paso enorme lleva los logits a valores gigantes, la siguiente actualización se dispara y aparece $\infty-\infty$.

**Ejercicio 6.** Escribe `entrenar_seguro(modelo, pasos=60, lr=10.0, max_norma=1.0)` igual que `entrenar` pero (a) recortando la norma global del gradiente con `nn.utils.clip_grad_norm_` antes de `opt.step()` y (b) deteniéndose con un aviso si la pérdida no es finita. Devuelve la lista de pérdidas.

In [ ]:
# TU CÓDIGO
def entrenar_seguro(modelo, pasos=60, lr=10.0, max_norma=1.0):
    return []

In [ ]:
# Comprobación
torch.manual_seed(1); h_seg = entrenar_seguro(ResidualSinNorma())
assert len(h_seg) == 60 and all(math.isfinite(v) for v in h_seg), "Con recorte la pérdida debe ser finita los 60 pasos"
print("Correcto. Máxima pérdida con recorte:", round(max(h_seg), 2), " sin recorte:", f"{max(v for v in h_roto if math.isfinite(v)):.2e}")
print("El recorte evita la explosión, pero lr = 10 sigue siendo mala idea: la solución de fondo es bajarla.")

## Parte 7 · Dropout invertido

En entrenamiento se anula cada unidad con probabilidad $p$ y se divide lo que queda entre $1-p$, para que la esperanza no cambie. En evaluación no se toca nada.

**Ejercicio 7.** Implementa `mi_dropout(x, p, entrenando=True)`.

In [ ]:
# TU CÓDIGO
def mi_dropout(x, p, entrenando=True):
    return x

In [ ]:
# Comprobación
torch.manual_seed(0)
x = torch.ones(200_000)
s = mi_dropout(x, 0.3)
assert abs(s.mean().item() - 1) < 0.01, "La media debe conservarse (divide entre 1 - p)"
assert abs((s == 0).float().mean().item() - 0.3) < 0.01, "Debe anular ~30 % de las unidades"
assert torch.equal(mi_dropout(x, 0.3, entrenando=False), x), "En evaluación debe devolver x"
print("Correcto. Valores distintos:", sorted(set(s[:20].tolist())))

## Para pensar

1. ¿Por qué el cociente de gradientes de la red plana se parece a $(1/\sqrt3)^{24}$ y no es exactamente igual?
2. Repite la Parte 2 con `torch.float8_e4m3fn`: ¿qué valor tiene 300?
3. Quita las normas de `RedResidual` y entrena con lr = 0,1: ¿aprende? ¿Y con lr = 1?

## Soluciones de referencia

Intenta los ejercicios antes de mirar. Descomenta lo que necesites.

In [ ]:
# --- ej1 ---
# def formato(e, m):
#     sesgo = 2**(e - 1) - 1
#     maximo = (2 - 2**-m) * 2**sesgo
#     eps = 2**-m
#     min_normal = 2**(1 - sesgo)
#     return maximo, eps, min_normal

# --- ej2 ---
# def lse(z):
#     c = z.max()
#     return c + torch.log(torch.exp(z - c).sum())

# --- ej3 ---
# def normas_grad(modelo):
#     modelo.zero_grad()
#     F.cross_entropy(modelo(X), y).backward()
#     return [c.weight.grad.norm().item() for c in modelo.capas]

# --- ej4 ---
# class MiRMSNorm(nn.Module):
#     def __init__(self, d, eps=1e-6):
#         super().__init__()
#         self.g = nn.Parameter(torch.ones(d))
#         self.eps = eps
#     def forward(self, x):
#         rms = torch.sqrt((x * x).mean(dim=-1, keepdim=True) + self.eps)
#         return self.g * x / rms

# --- ej5 ---
# class RedResidual(nn.Module):
#     def __init__(self):
#         super().__init__()
#         self.entrada = nn.Linear(D, W)
#         self.capas = nn.ModuleList([nn.Linear(W, W) for _ in range(L)])
#         self.normas = nn.ModuleList([MiRMSNorm(W) for _ in range(L)])
#         self.norma_final = MiRMSNorm(W)
#         self.salida = nn.Linear(W, C)
#     def forward(self, x):
#         h = self.entrada(x)
#         self.acts = []
#         for capa, norma in zip(self.capas, self.normas):
#             h = h + torch.tanh(capa(norma(h)))
#             self.acts.append(h)
#         return self.salida(self.norma_final(h))

# --- ej6 ---
# def entrenar_seguro(modelo, pasos=60, lr=10.0, max_norma=1.0):
#     opt = torch.optim.SGD(modelo.parameters(), lr=lr)
#     hist = []
#     for t in range(pasos):
#         loss = F.cross_entropy(modelo(X), y)
#         if not torch.isfinite(loss):
#             print("Pérdida no finita en el paso", t); break
#         opt.zero_grad()
#         loss.backward()
#         nn.utils.clip_grad_norm_(modelo.parameters(), max_norma)
#         opt.step()
#         hist.append(loss.item())
#     return hist

# --- ej7 ---
# def mi_dropout(x, p, entrenando=True):
#     if not entrenando or p == 0:
#         return x
#     mascara = (torch.rand_like(x) > p).float()
#     return x * mascara / (1 - p)